In [35]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    KFold,
    train_test_split,
    cross_val_score,
    GridSearchCV,
)

In [36]:
def apply_mpl_config():
    print("Apply custom plotting configs...")
    mpl_config = {
        "figure.figsize": (12, 6),
        "savefig.dpi": 300,
        "figure.dpi": 150,
        "font.family": "serif",
        "font.serif": ["Computer Modern"],
        "text.usetex": True,
        "font.size": 11,
        "axes.labelsize": 10,
        "axes.titlesize": 12,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 10,
        "lines.linewidth": 2,
        "axes.linewidth": 0.8,
        "savefig.bbox": "tight",
    }

    mpl.rcParams.update(mpl_config)
    print("Done!")

In [37]:
apply_mpl_config()

Apply custom plotting configs...
Done!


In [38]:
train = pd.read_csv("./train.csv")
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 9 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Cement_component1__kgInAM_3Mixture_            800 non-null    float64
 1   BlastFurnaceSlag_component2__kgInAM_3Mixture_  800 non-null    float64
 2   FlyAsh_component3__kgInAM_3Mixture_            800 non-null    float64
 3   Water_component4__kgInAM_3Mixture_             800 non-null    float64
 4   Superplasticizer_component5__kgInAM_3Mixture_  800 non-null    float64
 5   CoarseAggregate_component6__kgInAM_3Mixture_   800 non-null    float64
 6   FineAggregate_component7__kgInAM_3Mixture_     800 non-null    float64
 7   Age_day_                                       800 non-null    int64  
 8   ConcreteCompressiveStrength_MPa_Megapascals_   800 non-null    float64
dtypes: float64(8), int64(1)
memory usage: 56.4 KB


In [39]:
test = pd.read_csv("./test.csv")
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   Cement_component1__kgInAM_3Mixture_            100 non-null    float64
 1   BlastFurnaceSlag_component2__kgInAM_3Mixture_  100 non-null    float64
 2   FlyAsh_component3__kgInAM_3Mixture_            100 non-null    float64
 3   Water_component4__kgInAM_3Mixture_             100 non-null    float64
 4   Superplasticizer_component5__kgInAM_3Mixture_  100 non-null    float64
 5   CoarseAggregate_component6__kgInAM_3Mixture_   100 non-null    float64
 6   FineAggregate_component7__kgInAM_3Mixture_     100 non-null    float64
 7   Age_day_                                       100 non-null    int64  
 8   ConcreteCompressiveStrength_MPa_Megapascals_   100 non-null    float64
dtypes: float64(8), int64(1)
memory usage: 7.2 KB


In [40]:
train.head()

,Cement_component1__kgInAM_3Mixture_,BlastFurnaceSlag_component2__kgInAM_3Mixture_,FlyAsh_component3__kgInAM_3Mixture_,Water_component4__kgInAM_3Mixture_,Superplasticizer_component5__kgInAM_3Mixture_,CoarseAggregate_component6__kgInAM_3Mixture_,FineAggregate_component7__kgInAM_3Mixture_,Age_day_,ConcreteCompressiveStrength_MPa_Megapascals_
0,153.0,145.0,0.0,178.0,8.0,1000.0,822.0,28,19.008853
1,525.0,0.0,0.0,189.0,0.0,1125.0,613.0,14,48.401215
2,379.5,151.2,0.0,153.9,15.9,1134.3,605.0,3,28.599464
3,540.0,0.0,0.0,173.0,0.0,1125.0,613.0,90,69.657760
4,143.8,136.3,106.2,178.1,7.5,941.5,774.3,28,26.147688


In [41]:
test.columns

Index(['Cement_component1__kgInAM_3Mixture_',
       'BlastFurnaceSlag_component2__kgInAM_3Mixture_',
       'FlyAsh_component3__kgInAM_3Mixture_',
       'Water_component4__kgInAM_3Mixture_',
       'Superplasticizer_component5__kgInAM_3Mixture_',
       'CoarseAggregate_component6__kgInAM_3Mixture_',
       'FineAggregate_component7__kgInAM_3Mixture_', 'Age_day_',
       'ConcreteCompressiveStrength_MPa_Megapascals_'],
      dtype='str')

In [42]:
original_cols = [
    "Cement_component1__kgInAM_3Mixture_",
    "BlastFurnaceSlag_component2__kgInAM_3Mixture_",
    "FlyAsh_component3__kgInAM_3Mixture_",
    "Water_component4__kgInAM_3Mixture_",
    "Superplasticizer_component5__kgInAM_3Mixture_",
    "CoarseAggregate_component6__kgInAM_3Mixture_",
    "FineAggregate_component7__kgInAM_3Mixture_",
    "Age_day_",
    "ConcreteCompressiveStrength_MPa_Megapascals_",
]

In [43]:
simple_cols = [
    "Cement",
    "BlastFurnaceSlag",
    "FlyAsh",
    "Water",
    "Superplasticizer",
    "CoarseAggregate",
    "FineAggregate",
    "Age_day",
    "ConcCompStrength_MPa",
]

In [44]:
# Rename columns in train and test
train.columns = simple_cols
test.columns = simple_cols

In [45]:
train.head()

,Cement,BlastFurnaceSlag,FlyAsh,Water,Superplasticizer,CoarseAggregate,FineAggregate,Age_day,ConcCompStrength_MPa
0,153.0,145.0,0.0,178.0,8.0,1000.0,822.0,28,19.008853
1,525.0,0.0,0.0,189.0,0.0,1125.0,613.0,14,48.401215
2,379.5,151.2,0.0,153.9,15.9,1134.3,605.0,3,28.599464
3,540.0,0.0,0.0,173.0,0.0,1125.0,613.0,90,69.657760
4,143.8,136.3,106.2,178.1,7.5,941.5,774.3,28,26.147688


In [46]:
X_train = train.drop("ConcCompStrength_MPa", axis=1)
y_train = train["ConcCompStrength_MPa"]

X_test_holdout = test.drop("ConcCompStrength_MPa", axis=1)
y_test_holdout = test["ConcCompStrength_MPa"]

In [47]:
def calc_rse(y_test, y_pred, p):
    """
    Calculate RSE - Residual Standard Error

    Args:
        y_test: test set
        y_pred: predicted set
        p: number of features

    Returns RSE
    """
    n = len(y_test)
    df = n - p - 1

    assert n == len(y_pred), "Prediction and target length mismatch"
    assert n > p + 1, f"Insufficient samples: need n > {p+1}, got {n}"

    residuals = y_test - y_pred
    rss = np.sum(residuals**2)

    return np.sqrt(rss / df)


def calc_r2(y_test, y_pred):
    """
    Calculate R^2 - Coefficient of determination

    Args:
        Args:
        y_test: test set
        y_pred: predicted set

    Returns R^2
    """
    assert len(y_test) == len(y_pred), "Prediction and target length mismatch"

    ss_res = np.sum((y_test - y_pred) ** 2)  # sum of squares residuals
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)  # sum of squares total

    return 1 - (ss_res / ss_tot)

## Question 1


In [48]:
# define scaler
scaler = StandardScaler()

# Define linear regression model
reg = LinearRegression()

# Define Pipeline
pipe_lr = Pipeline([("scaler", scaler), ("reg", reg)])

In [49]:
# Split: train 60, val 20, test 20

# Split 1: train 100 -> train_temp 80, train_test 20
X_train_temp, X_train_test, y_train_temp, y_train_test = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, shuffle=True
)

# Split 2: train_temp 80 -> train_train 60, train_val 20
X_train_train, X_train_val, y_train_train, y_train_val = train_test_split(
    X_train_temp, y_train_temp, test_size=0.25, random_state=42, shuffle=True
)

print(f"Train size: {len(X_train_train)} ({len(X_train_train)/len(X_train)*100:.1f}%)")
print(f"Validation size: {len(X_train_val)} ({len(X_train_val)/len(X_train)*100:.1f}%)")
print(f"Test size: {len(X_train_test)} ({len(X_train_test)/len(X_train)*100:.1f}%)")

Train size: 480 (60.0%)
Validation size: 160 (20.0%)
Test size: 160 (20.0%)


In [50]:
# Combine train + val
X_train_full = np.vstack([X_train_train, X_train_val])
y_train_full = np.hstack([y_train_train, y_train_val])

# Train on train + val (full)
pipe_lr.fit(X_train_full, y_train_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None


In [51]:
# Predict on test (train test subset)
y_pred_val = pipe_lr.predict(X_train_test.values)

p = X_train.shape[1]  # n_features

# Calc errors
rse_val = calc_rse(y_train_test, y_pred_val, p)
r2_val = calc_r2(y_train_test, y_pred_val)

print(f"RSE:  {rse_val:.4f}")
print(f"R^2:   {r2_val:.4f}")

RSE:  11.6501
R^2:   0.5445


### Cross-validation


In [52]:
n_folds = {5, 10, 15}  # number of folds

for n_fold in n_folds:
    print(f"\nFor CV n-folds={n_fold}")
    cv = KFold(n_splits=n_fold, shuffle=True, random_state=42)
    mse_scores = -cross_val_score(
        pipe_lr, X_train_full, y_train_full, cv=cv, scoring="neg_mean_squared_error"
    )
    rmse_scores = np.sqrt(mse_scores)

    # Average RMSE across folds
    rmse_cv = np.mean(rmse_scores)
    rmse_cv_std = np.std(rmse_scores)

    # print(f"\nCross-Validation Results:")
    print(f"  RMSE (mean): {rmse_cv:.4f} +- {rmse_cv_std:.4f}")
    print(f"  RMSE (min):  {np.min(rmse_scores):.4f}")
    print(f"  RMSE (max):  {np.max(rmse_scores):.4f}")

    # For RSE in CV: approximate using full dataset fit
    pipe_lr.fit(X_train, y_train)
    y_pred_full = pipe_lr.predict(X_train)
    rse_cv_approx = calc_rse(y_train, y_pred_full, p)
    print(f"  RSE (approx from full fit): {rse_cv_approx:.4f}")


For CV n-folds=10


  RMSE (mean): 10.1779 +- 1.0202
  RMSE (min):  8.8006
  RMSE (max):  12.2935
  RSE (approx from full fit): 10.2647

For CV n-folds=5
  RMSE (mean): 10.2568 +- 0.7953
  RMSE (min):  9.1789
  RMSE (max):  11.5419
  RSE (approx from full fit): 10.2647

For CV n-folds=15
  RMSE (mean): 10.0985 +- 1.2204
  RMSE (min):  7.9154
  RMSE (max):  12.8818
  RSE (approx from full fit): 10.2647


## Question 2


In [53]:
# Ridge hyperparameter tuning
ridge = Ridge()
pipe_ridge = Pipeline([("scaler", scaler), ("ridge", ridge)])

# define grid
alphas = np.logspace(-3, 3, 50)  # 0.001 to 1000
param_grid = {"ridge__alpha": alphas}

print(f"Testing {len(alphas)} alpha values from {alphas[0]:.4f} to {alphas[-1]:.2f}")

# Grid search
cv = KFold(n_splits=10, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipe_ridge,
    param_grid,
    cv=cv,
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

# Extract results
best_alpha_ridge = grid_search.best_params_["ridge__alpha"]
cv_results = grid_search.cv_results_

print(f"\nBest alpha: {best_alpha_ridge:.6f}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Testing 50 alpha values from 0.0010 to 1000.00
Fitting 10 folds for each of 50 candidates, totalling 500 fits

Best alpha: 1.526418
Best CV RMSE: 10.3534


In [68]:
pipe_ridge_full = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=best_alpha_ridge, max_iter=10000)),
    ]
)

# Train on train + val (full)
pipe_ridge_full.fit(X_train_full, y_train_full)

# Predict on test (train test subset)
y_pred_ridge = pipe_ridge_full.predict(X_train_test.values)

# Calc errors
rse_ridge = calc_rse(y_train_test, y_pred_ridge, p)
r2_ridge = calc_r2(y_train_test, y_pred_ridge)

print(f"RSE:  {rse_ridge:.4f}")
print(f"R^2:   {r2_ridge:.4f}")

RSE:  11.6520
R^2:   0.5443


In [55]:
# Compare simple LR and Ridge

## Question 3


In [71]:
# Lasso hyperparameter tuning
lasso = Lasso()
pipe_lasso = Pipeline([("scaler", scaler), ("lasso", lasso)])

# define grid
alphas = np.logspace(-3, 3, 50)  # 0.001 to 1000
param_grid = {"lasso__alpha": alphas}

print(f"Testing {len(alphas)} alpha values from {alphas[0]:.4f} to {alphas[-1]:.2f}")

# Grid search
cv = KFold(n_splits=10, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipe_lasso,
    param_grid,
    cv=cv,
    scoring="neg_mean_squared_error",
    return_train_score=True,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

# Extract results
best_alpha_lasso = grid_search.best_params_["lasso__alpha"]
cv_results = grid_search.cv_results_

print(f"\nBest alpha: {best_alpha_lasso:.6f}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Testing 50 alpha values from 0.0010 to 1000.00
Fitting 10 folds for each of 50 candidates, totalling 500 fits

Best alpha: 0.068665
Best CV RMSE: 10.3470


In [72]:
pipe_lasso_full = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=best_alpha_lasso, max_iter=10000)),
    ]
)

# Train on train + val (full)
pipe_lasso_full.fit(X_train_full, y_train_full)

# Predict on test (train test subset)
y_pred_lasso = pipe_lasso_full.predict(X_train_test.values)

# Calc errors
rse_lasso = calc_rse(y_train_test, y_pred_lasso, p)
r2_lasso = calc_r2(y_train_test, y_pred_lasso)

print(f"RSE:  {rse_lasso:.4f}")
print(f"R^2:   {r2_lasso:.4f}")

RSE:  11.6344
R^2:   0.5457
